# Human vs Machine Translation Detection Codebase

This Jupyter Notebook is part of a project focused on detecting whether translations are generated by humans or machines. It provides a workflow for generating machine translations using the OpenAI API, which can be used to create datasets for training and evaluating translation detection models.

## Key Components:
- **Setup and Configuration**: Imports necessary libraries (e.g., OpenAI, pandas, tqdm) and initializes the OpenAI client with API key and base URL.
- **Prompt Creation**: Defines a function to generate prompts for translating sentences from a source language to a target language.
- **API Interaction**: Implements a function to call the OpenAI API with retry logic, handling errors and rate limiting.
- **Data Loading**: Includes functions to read sentences from text files or CSV files, supporting batch processing.
- **Translation Process**: Reads source sentences, chunks them if needed, and translates them using the API, with progress tracking.
- **Output**: Saves the translated sentences to a specified output file for further analysis or model training.

This notebook facilitates the collection of machine-generated translations, which can be paired with human translations to build a detection system. Ensure to set API keys, file paths, and parameters appropriately before execution.

In [ ]:
import openai
import pandas as pd
import csv
import time
from typing import Dict, Tuple
import os

# Configuration
API_KEY = "" # Set this to your OpenAI API key
BASE_URL = "" # Set this to your OpenAI base URL

# Initialize OpenAI client
client = openai.OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)


In [2]:
def create_prompt(sentence: str,source_lang,target_lang) -> str:
    """Create the classification prompt for OpenAI."""
    prompt = f"""
    You are an expert {target_lang} translator. Your task is to translate the given {source_lang} sentence into {target_lang}. You mush output only the translated sentence without any additional text or explanations.
    ## Sentence
    {sentence}
    """

    return prompt


In [ ]:
def call_openai_api(client: openai.OpenAI, sentence: str, max_retries: int = 3,source_lang: str = "French",target_lang="English") -> Tuple[str, str]:
    """
    Call OpenAI API and parse the response.
    Returns: (label, reason)
    """
    prompt = create_prompt(sentence,source_lang,target_lang=target_lang)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="claude-opus-4-20250514-v1:0", # Use the appropriate model name, such as gpt5
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.0  # For more consistent results
            )
            
            # Extract the response text
            response_text = response.choices[0].message.content.strip()
            # print(response_text)
            
            return "TRANSLATED", response_text
            
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff
                print(f"Error on attempt {attempt + 1}: {e}. Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                return "ERROR", f"API Error: {str(e)}"
    
    return "ERROR", "Max retries exceeded"


In [12]:
def readTxtFiles(source_file):
    try:
        with open(source_file, "r", encoding="utf-8") as f:
            original_sentences = f.readlines()
            # Remove any leading/trailing whitespace from each line
            original_sentences = [line.strip() for line in original_sentences if line.strip()]
            
        print(f"Read {len(original_sentences)} sentences from file.")
        print("-" * 30)
        return original_sentences
    except FileNotFoundError:
        print("Error: file not found.")
        exit()

In [4]:
def readCsvFile(source_file, col_name="source text"):
    try:
        sentences = []

        with open(source_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)

            if not reader.fieldnames or col_name not in reader.fieldnames:
                raise ValueError(f"Column '{col_name}' not found in CSV.")

            for row in reader:
                sentence = row.get(col_name, "").strip()
                if sentence:
                    sentences.append(sentence)

        print(f"Read {len(sentences)} sentences from CSV.")
        print("-" * 30)
        return sentences

    except FileNotFoundError:
        raise FileNotFoundError("Error: file not found.")

In [ ]:
source_files=[] # Set this to the list of txt source files that contain the sentences to be translated

original_sentences=[]
for source in source_files:
    original_sentences.append(readTxtFiles(source))

    print(len(original_sentences[-1]))

In [ ]:
source_file = "" # Set this to the path of the input CSV file that contains the sentences to be chunked 
sentences = readCsvFile(source_file)

# Create list of lists 
chunk_size = 200
original_sentences = [
    sentences[i:i + chunk_size] 
    for i in range(0, len(sentences), chunk_size)
]

print(f"Total chunks created: {len(original_sentences)}")


Read 596 sentences from CSV.
------------------------------
Total chunks created: 3


In [ ]:
from tqdm import tqdm

translations = []
orginal_languages=[] # Set this to the list of original languages corresponding to each source file, e.g., ["English", "French", "German"]
target_language = "" # Set this to the target language for translation, e.g., "Spanish"
sentence_number = # Set this to the number of sentences to translate from each source file, e.g., 5000
for j,original_language in enumerate(orginal_languages):
    start=0
    for i, sentence in enumerate(tqdm(original_sentences[j][start:start+sentence_number], desc="Translating")):
        
        sentence = sentence.strip()
        if sentence:
            status, translated_text = call_openai_api(client, sentence, max_retries=3,source_lang=original_language,target_lang=target_language)
            translations.append( translated_text)

            # print(translated_text)

            # Delay to avoid rate limiting
            time.sleep(1)  
        else:
            print("Translation Failed")
        

print("All sentences translated successfully.")
print("-" * 30)

In [ ]:
print("Translations:")
for i, translation in enumerate(translations[:5]):
    print(f"{i+1}. {translation}")
print("-" * 30)
print(len(translations))

In [ ]:
output_filename =  "" # Set this to the desired output filename for the translations, e.g., "translated_sentences.txt"
try:
    with open(output_filename, "w", encoding="utf-8") as f:
        for translation in translations:
            cleaned = translation.replace("\n", " ").replace("\r", " ")
            f.write(cleaned + "\n")
    print(f"Translations successfully saved to '{output_filename}'.")
    print("-" * 30)
except Exception as e:
    print(f"Error saving translations: {e}")
    exit()